In [1]:
print("Non-IID split completed.")

Non-IID split completed.


## Cell 0 — Seed everything (important)

In [2]:
import os, random
import numpy as np
import torch

SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
print("Seed fixed:", SEED)

Seed fixed: 42


## Cell 1 — Setup (same as before)

In [3]:
from pathlib import Path
import sys
import torch
from torch.utils.data import DataLoader, Subset
import torch.optim as optim
import numpy as np

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "notebooks" else cwd
sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset import Kits2DSegDataset
from src.transforms import TransformConfig, SegTransform
from src.models_resnet_unet import ResNet18UNet
from src.train_seg import TrainConfig, train_one_epoch, evaluate
from src.federated.partition import (
    group_indices_by_case, iid_case_split, build_client_indices_from_cases,
    noniid_split_by_tumor_burden
)
from src.federated.fedavg import get_state_dict, set_state_dict, fedavg

DATA_ROOT = Path(r"F:\projects\hirdl\FedSSL_Paper\data_set\kits_2d_splitted")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

tfm = SegTransform(TransformConfig(out_size=256, to_3ch=True, normalize_01=True))
train_ds = Kits2DSegDataset(DATA_ROOT, "train", transform=tfm)
val_ds   = Kits2DSegDataset(DATA_ROOT, "val", transform=tfm)
test_ds  = Kits2DSegDataset(DATA_ROOT, "test", transform=tfm)

BATCH = 8
NUM_WORKERS = 0  # keep 0 for Windows stability

Device: cuda


In [4]:
from pathlib import Path
p = Path(r"f:\projects\hirdl\FedSSL_Paper\src\federated\partition.py")
print("partition.py exists:", p.exists())
print("Last 60 lines:\n")
print("\n".join(p.read_text(encoding="utf-8").splitlines()[-60:]))

partition.py exists: True
Last 60 lines:

# src/federated/partition.py
from __future__ import annotations
from collections import defaultdict
from typing import Dict, List, Tuple
import re
import random

CASE_RE = re.compile(r"(case_\d{5})", re.IGNORECASE)

def extract_case_id(filename: str) -> str:
    m = CASE_RE.search(filename)
    if not m:
        raise ValueError(f"Could not extract case id from: {filename}")
    return m.group(1).lower()

def group_indices_by_case(meta_filename_list: List[str]) -> Dict[str, List[int]]:
    cases = defaultdict(list)
    for idx, fn in enumerate(meta_filename_list):
        cid = extract_case_id(fn)
        cases[cid].append(idx)
    return dict(cases)

def iid_case_split(case_ids: List[str], num_clients: int, seed: int = 42) -> List[List[str]]:
    rng = random.Random(seed)
    case_ids = list(case_ids)
    rng.shuffle(case_ids)
    return [case_ids[i::num_clients] for i in range(num_clients)]

def build_client_indices_from_cases(case_to_indices

## Cell 2 — Build case mapping once

In [5]:
train_filenames = [train_ds[i][2]["filename"] for i in range(len(train_ds))]
case_to_idxs = group_indices_by_case(train_filenames)
case_ids = sorted(case_to_idxs.keys())
print("Train cases:", len(case_ids))

Train cases: 342


## Cell 3 — Compute tumor burden per case (non-IID driver)

In [6]:
from PIL import Image
from tqdm import tqdm

def tumor_fraction_from_mask(mask_arr: np.ndarray) -> float:
    # mask classes after transform are 0/1/2, but here we load raw mask png: 0/127/255
    tumor = (mask_arr == 255).sum()
    return float(tumor) / float(mask_arr.size)

# Build map filename->index for quick lookup
filename_to_index = {train_ds[i][2]["filename"]: i for i in range(len(train_ds))}

# Build mask path lookup from dataset pairs
# (train_ds.pairs exists in our dataset class)
pairs = train_ds.pairs  # list of (img_path, mask_path)
mask_by_filename = {ip.name: mp for ip, mp in pairs}

case_burden = {}
for cid, idxs in tqdm(case_to_idxs.items(), total=len(case_to_idxs)):
    fracs = []
    for i in idxs:
        fn = train_filenames[i]
        mp = mask_by_filename[fn]
        mk = np.array(Image.open(mp))
        fracs.append(tumor_fraction_from_mask(mk))
    case_burden[cid] = float(np.mean(fracs))

# quick summary
vals = np.array(list(case_burden.values()))
print("Burden stats:", "min", vals.min(), "median", np.median(vals), "p95", np.percentile(vals,95), "max", vals.max())

100%|██████████| 342/342 [00:05<00:00, 57.29it/s]


Burden stats: min 0.000392913818359375 median 0.004529062906901042 p95 0.03507659639648094 max 0.15115711905739523


## Cell 4 — Helper: run one FedAvg experiment

In [7]:
from pathlib import Path

SIMCLR_ENCODER = Path("outputs/checkpoints/simclr_resnet18_encoder.pt")

def make_loaders_from_cases(client_cases, batch=BATCH):
    client_indices = build_client_indices_from_cases(case_to_idxs, client_cases)
    client_loaders, client_sizes = [], []
    for idxs in client_indices:
        subset = Subset(train_ds, idxs)
        loader = DataLoader(subset, batch_size=batch, shuffle=True,
                            num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=="cuda"))
        client_loaders.append(loader)
        client_sizes.append(len(subset))

    val_loader = DataLoader(val_ds, batch_size=batch, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=="cuda"))
    test_loader = DataLoader(test_ds, batch_size=batch, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=="cuda"))
    return client_loaders, client_sizes, val_loader, test_loader

def run_fedavg_experiment(name: str, client_cases, use_simclr_init: bool,
                         rounds: int = 10, local_epochs: int = 1):
    cfg = TrainConfig(epochs=local_epochs, lr=3e-4, dice_weight=0.5, use_amp=True)

    client_loaders, client_sizes, val_loader, test_loader = make_loaders_from_cases(client_cases)

    global_model = ResNet18UNet(num_classes=3).to(DEVICE)
    if use_simclr_init:
        assert SIMCLR_ENCODER.exists(), f"Missing {SIMCLR_ENCODER}"
        info = global_model.load_simclr_encoder(str(SIMCLR_ENCODER))
        print(f"[{name}] loaded SimCLR:", info)

    global_state = get_state_dict(global_model)
    hist = []

    for rnd in range(1, rounds + 1):
        client_states, weights = [], []
        for c, loader in enumerate(client_loaders):
            client_model = ResNet18UNet(num_classes=3).to(DEVICE)
            set_state_dict(client_model, global_state)

            opt = optim.AdamW(client_model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
            scaler = torch.cuda.amp.GradScaler(enabled=(cfg.use_amp and DEVICE.type=="cuda"))

            _ = train_one_epoch(client_model, loader, opt, DEVICE, cfg, scaler=scaler)

            client_states.append(get_state_dict(client_model))
            weights.append(client_sizes[c])

        global_state = fedavg(client_states, weights)
        set_state_dict(global_model, global_state)

        va = evaluate(global_model, val_loader, DEVICE, cfg)
        hist.append({"round": rnd, **va})
        print(f"[{name}] Round {rnd:02d} | val_tumor_dice={va['tumor_dice']:.4f} val_loss={va['val_loss']:.4f}")

    te = evaluate(global_model, test_loader, DEVICE, cfg)
    print(f"\n[{name}] TEST:", te)
    return hist, te

## Cell 5 — Run IID experiments (supervised vs SimCLR init)

In [8]:
NUM_CLIENTS = 3
ROUNDS = 10
LOCAL_EPOCHS = 1

# IID split
client_cases_iid = iid_case_split(case_ids, num_clients=NUM_CLIENTS, seed=SEED)

hist_iid_sup, test_iid_sup = run_fedavg_experiment(
    name="IID FedAvg supervised",
    client_cases=client_cases_iid,
    use_simclr_init=False,
    rounds=ROUNDS,
    local_epochs=LOCAL_EPOCHS,
)

hist_iid_ssl, test_iid_ssl = run_fedavg_experiment(
    name="IID FedAvg + SimCLR init",
    client_cases=client_cases_iid,
    use_simclr_init=True,
    rounds=ROUNDS,
    local_epochs=LOCAL_EPOCHS,
)

C:\Users\user\AppData\Local\Temp\ipykernel_8080\2760007474.py:43: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(cfg.use_amp and DEVICE.type=="cuda"))


[IID FedAvg supervised] Round 01 | val_tumor_dice=0.0000 val_loss=0.6365
[IID FedAvg supervised] Round 02 | val_tumor_dice=0.4053 val_loss=0.3483
[IID FedAvg supervised] Round 03 | val_tumor_dice=0.3059 val_loss=0.3960
[IID FedAvg supervised] Round 04 | val_tumor_dice=0.3436 val_loss=0.3752
[IID FedAvg supervised] Round 05 | val_tumor_dice=0.3116 val_loss=0.3973
[IID FedAvg supervised] Round 06 | val_tumor_dice=0.3802 val_loss=0.3566
[IID FedAvg supervised] Round 07 | val_tumor_dice=0.3655 val_loss=0.3658
[IID FedAvg supervised] Round 08 | val_tumor_dice=0.3481 val_loss=0.3783
[IID FedAvg supervised] Round 09 | val_tumor_dice=0.3347 val_loss=0.3851
[IID FedAvg supervised] Round 10 | val_tumor_dice=0.3495 val_loss=0.3806

[IID FedAvg supervised] TEST: {'val_loss': 0.4596071813016848, 'tumor_dice': 0.35681552979123604, 'tumor_iou': 0.3092867806276376, 'kidney_dice': 0.7658036954724843, 'kidney_iou': 0.6871394922948926}


f:\projects\hirdl\FedSSL_Paper\src\models_resnet_unet.py:95: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(simclr_encoder_ckpt, map_location="cpu")


[IID FedAvg + SimCLR init] loaded SimCLR: {'missing': [], 'unexpected': []}
[IID FedAvg + SimCLR init] Round 01 | val_tumor_dice=0.0002 val_loss=0.6484
[IID FedAvg + SimCLR init] Round 02 | val_tumor_dice=0.1943 val_loss=0.4592
[IID FedAvg + SimCLR init] Round 03 | val_tumor_dice=0.4121 val_loss=0.3366
[IID FedAvg + SimCLR init] Round 04 | val_tumor_dice=0.3472 val_loss=0.3771
[IID FedAvg + SimCLR init] Round 05 | val_tumor_dice=0.3449 val_loss=0.3757
[IID FedAvg + SimCLR init] Round 06 | val_tumor_dice=0.3892 val_loss=0.3504
[IID FedAvg + SimCLR init] Round 07 | val_tumor_dice=0.3717 val_loss=0.3654
[IID FedAvg + SimCLR init] Round 08 | val_tumor_dice=0.3226 val_loss=0.3903
[IID FedAvg + SimCLR init] Round 09 | val_tumor_dice=0.3688 val_loss=0.3627
[IID FedAvg + SimCLR init] Round 10 | val_tumor_dice=0.3506 val_loss=0.3786

[IID FedAvg + SimCLR init] TEST: {'val_loss': 0.4178116463062142, 'tumor_dice': 0.37829847050267257, 'tumor_iou': 0.3217059710877656, 'kidney_dice': 0.773487442863

## Cell 6 — Run non-IID experiments (supervised vs SimCLR init)

In [9]:
# non-IID split by tumor burden
client_cases_noniid = noniid_split_by_tumor_burden(case_ids, case_burden, num_clients=NUM_CLIENTS, seed=SEED)

hist_ni_sup, test_ni_sup = run_fedavg_experiment(
    name="Non-IID FedAvg supervised",
    client_cases=client_cases_noniid,
    use_simclr_init=False,
    rounds=ROUNDS,
    local_epochs=LOCAL_EPOCHS,
)

hist_ni_ssl, test_ni_ssl = run_fedavg_experiment(
    name="Non-IID FedAvg + SimCLR init",
    client_cases=client_cases_noniid,
    use_simclr_init=True,
    rounds=ROUNDS,
    local_epochs=LOCAL_EPOCHS,
)

C:\Users\user\AppData\Local\Temp\ipykernel_8080\2760007474.py:43: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(cfg.use_amp and DEVICE.type=="cuda"))


[Non-IID FedAvg supervised] Round 01 | val_tumor_dice=0.0106 val_loss=0.6423
[Non-IID FedAvg supervised] Round 02 | val_tumor_dice=0.1824 val_loss=0.4589
[Non-IID FedAvg supervised] Round 03 | val_tumor_dice=0.2889 val_loss=0.4051
[Non-IID FedAvg supervised] Round 04 | val_tumor_dice=0.2529 val_loss=0.4225
[Non-IID FedAvg supervised] Round 05 | val_tumor_dice=0.3194 val_loss=0.3902
[Non-IID FedAvg supervised] Round 06 | val_tumor_dice=0.2996 val_loss=0.4024
[Non-IID FedAvg supervised] Round 07 | val_tumor_dice=0.3181 val_loss=0.3930
[Non-IID FedAvg supervised] Round 08 | val_tumor_dice=0.2923 val_loss=0.4047
[Non-IID FedAvg supervised] Round 09 | val_tumor_dice=0.2586 val_loss=0.4254
[Non-IID FedAvg supervised] Round 10 | val_tumor_dice=0.2558 val_loss=0.4263

[Non-IID FedAvg supervised] TEST: {'val_loss': 0.45986837787287577, 'tumor_dice': 0.3105111407309839, 'tumor_iou': 0.2658872911736, 'kidney_dice': 0.7555060314690316, 'kidney_iou': 0.6757424277612248}
[Non-IID FedAvg + SimCLR ini

f:\projects\hirdl\FedSSL_Paper\src\models_resnet_unet.py:95: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(simclr_encoder_ckpt, map_location="cpu")


[Non-IID FedAvg + SimCLR init] Round 01 | val_tumor_dice=0.0000 val_loss=0.6407
[Non-IID FedAvg + SimCLR init] Round 02 | val_tumor_dice=0.3465 val_loss=0.3752
[Non-IID FedAvg + SimCLR init] Round 03 | val_tumor_dice=0.4476 val_loss=0.3205
[Non-IID FedAvg + SimCLR init] Round 04 | val_tumor_dice=0.4340 val_loss=0.3284
[Non-IID FedAvg + SimCLR init] Round 05 | val_tumor_dice=0.4301 val_loss=0.3288
[Non-IID FedAvg + SimCLR init] Round 06 | val_tumor_dice=0.4233 val_loss=0.3338
[Non-IID FedAvg + SimCLR init] Round 07 | val_tumor_dice=0.4135 val_loss=0.3400
[Non-IID FedAvg + SimCLR init] Round 08 | val_tumor_dice=0.4311 val_loss=0.3326
[Non-IID FedAvg + SimCLR init] Round 09 | val_tumor_dice=0.3912 val_loss=0.3532
[Non-IID FedAvg + SimCLR init] Round 10 | val_tumor_dice=0.3945 val_loss=0.3501

[Non-IID FedAvg + SimCLR init] TEST: {'val_loss': 0.4326502591402841, 'tumor_dice': 0.39756511477320866, 'tumor_iou': 0.3406222509153743, 'kidney_dice': 0.7817846092945395, 'kidney_iou': 0.7014799930

## Cell 7 — Summarize results in one table

In [10]:
import pandas as pd

summary = [
    {"setting":"Federated IID", "method":"FedAvg supervised", "test_tumor_dice": test_iid_sup["tumor_dice"], "test_tumor_iou": test_iid_sup["tumor_iou"]},
    {"setting":"Federated IID", "method":"FedAvg + SimCLR init", "test_tumor_dice": test_iid_ssl["tumor_dice"], "test_tumor_iou": test_iid_ssl["tumor_iou"]},
    {"setting":"Federated non-IID", "method":"FedAvg supervised", "test_tumor_dice": test_ni_sup["tumor_dice"], "test_tumor_iou": test_ni_sup["tumor_iou"]},
    {"setting":"Federated non-IID", "method":"FedAvg + SimCLR init", "test_tumor_dice": test_ni_ssl["tumor_dice"], "test_tumor_iou": test_ni_ssl["tumor_iou"]},
]
pd.DataFrame(summary)

,setting,method,test_tumor_dice,test_tumor_iou
0,Federated IID,FedAvg supervised,0.356816,0.309287
1,Federated IID,FedAvg + SimCLR init,0.378298,0.321706
2,Federated non-IID,FedAvg supervised,0.310511,0.265887
3,Federated non-IID,FedAvg + SimCLR init,0.397565,0.340622
